# Visualización de partidos, mapas y equipos

Este notebook **no re-scrapea nada**: usa los datos ya scrapeados (Capa 1)
que están en `data/interim/`. Reemplaza al antiguo `etl_match_details`.

Muestra: mapas más jugados, win rate por mapa, marcadores más frecuentes,
top equipos por win rate, y ataque vs defensa por mapa.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # fuera de Colab (local)


In [ ]:
import sys
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Analisis de datos/Proyecto final")
if not ROOT.exists():
    ROOT = Path.cwd()  # fallback local
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))

from config import get_data_dir
DATA_DIR = str(get_data_dir())

from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
print("DATA_DIR =", DATA_DIR)


In [ ]:
# Carga los datos ya scrapeados (Capa 1). No se llama a la API.
team_stats = pd.read_parquet(f"{DATA_DIR}/interim/team_stats.parquet")
matches = pd.read_parquet(f"{DATA_DIR}/interim/matches.parquet")
rankings = pd.read_parquet(f"{DATA_DIR}/interim/rankings.parquet")
teams = pd.read_parquet(f"{DATA_DIR}/interim/teams.parquet")

# win_pct viene como "67%": convertir a fracción 0-1
team_stats["win_pct_num"] = (
    pd.to_numeric(team_stats["win_pct"].astype(str).str.replace("%", "", regex=False),
                  errors="coerce") / 100.0
)

print("team_stats:", team_stats.shape, "| matches:", matches.shape, "| rankings:", rankings.shape)


In [ ]:
# 1) Mapas más jugados (partidas registradas)
games_by_map = team_stats.groupby("map")["games"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
games_by_map.plot(kind="bar", color="#4c8bf5", ax=ax)
ax.set_title("Partidas registradas por mapa")
ax.set_ylabel("Partidas")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 2) Win rate promedio por mapa (entre todos los equipos)
win_by_map = team_stats.groupby("map")["win_pct_num"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
(win_by_map * 100).plot(kind="bar", color="#34a853", ax=ax)
ax.set_title("Win rate promedio por mapa")
ax.set_ylabel("Win rate (%)")
ax.set_xlabel("")
ax.axhline(50, color="gray", linestyle="--", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Win rate promedio por mapa (%):")
print((win_by_map * 100).round(1).to_string())


In [ ]:
# 3) Marcadores más frecuentes (score1-score2)
scores = Counter(zip(matches["score1"].fillna(-1), matches["score2"].fillna(-1)))
top_scores = pd.Series(scores).sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(11, 5))
top_scores.plot(kind="bar", color="#fbbc04", ax=ax)
ax.set_title("Marcadores más frecuentes")
ax.set_xlabel("(score1, score2)")
ax.set_ylabel("Partidos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 4) Top equipos por win rate (mínimo 30 partidos), desde matches
wins = Counter()
games = Counter()
for r in matches.itertuples():
    games[r.team1] += 1
    games[r.team2] += 1
    if r.winner == "team1":
        wins[r.team1] += 1
    elif r.winner == "team2":
        wins[r.team2] += 1

wr = pd.Series({t: wins[t] / games[t] for t in games if games[t] >= 30}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
wr.head(12).plot(kind="barh", color="#34a853", ax=ax)
ax.set_title("Top 12 equipos por win rate (mín. 30 partidos)")
ax.set_xlabel("Win rate")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# 5) Ataque vs defensa por mapa (round win rate promedio)
atk = team_stats.groupby("map").apply(
    lambda g: pd.to_numeric(g["atk_rwin_pct"].astype(str).str.replace("%", "", regex=False),
                            errors="coerce").mean() / 100.0)
dfn = team_stats.groupby("map").apply(
    lambda g: pd.to_numeric(g["def_rwin_pct"].astype(str).str.replace("%", "", regex=False),
                            errors="coerce").mean() / 100.0)

side = pd.DataFrame({"Ataque": atk, "Defensa": dfn}).sort_values("Ataque")
side.plot(kind="bar", figsize=(11, 5), color=["#ea4335", "#4285f4"])
plt.title("Round win rate: ataque vs defensa por mapa")
plt.ylabel("Round win rate")
plt.xlabel("Mapa")
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Lectura rápida

- **Haven, Lotus, Ascent** son los mapas más jugados.
- El win rate por mapa ronda el **50%** (son enfrentamientos balanceados), con
  pequeñas diferencias según el mapa.
- El marcador dominante es **2-0** (el doble que 2-1), señal de que la dominancia
  es informativa.
- Algunos mapas favorecen más a **defensa** que a **ataque** (o viceversa).
